# V11 Foundation-Model — Chronos zero-shot on Google Colab

Run Amazon's Chronos-T5-Small (Ansari et al., 2024) on each
`(Партнер, Артикул)` monthly shipment series and produce
**zero-shot** forecasts for the validation + test windows.
Chronos was pre-trained on M-series competitions, electricity,
traffic, weather and finance — **never seen toy-distribution data**
— so its residuals are guaranteed orthogonal to V1–V10.

## What you need

1. A **Google Drive** account (free 15 GB tier is enough).
2. The file `output/abt_v10_cached.parquet` from this repo
   (≈ 75 MB — copy it to Drive yourself).
3. A Colab runtime with a **T4 GPU** (free tier: Runtime → Change
   runtime type → T4 GPU).

## What this notebook does

Cell 1: install Chronos
Cell 2: mount Drive, locate the ABT
Cell 3: build per-pair monthly history
Cell 4: load Chronos-T5-Small weights (~200 MB)
Cell 5: zero-shot forecasting (~25 min on T4)
Cell 6: write `preds_v11_chronos_val.csv` + `_test.csv` to Drive
Cell 7: print quick WAPE / bias metrics

## After it finishes

Download both CSVs from your Drive into `output/` of this repo
and re-run `python -m scripts.v11_lad_stack`.  Chronos appears
automatically in the LAD pool.

In [ ]:
%pip install --quiet chronos-forecasting==1.5.2
%pip install --quiet --upgrade transformers tokenizers
import sys, os, glob, time
import numpy as np, pandas as pd, torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('CUDA cap.:', torch.cuda.get_device_capability(0))

In [ ]:
# Mount Google Drive and locate the ABT file
from google.colab import drive
drive.mount('/content/drive')

# Adjust this path to wherever you put abt_v10_cached.parquet on Drive
ABT_PATH = '/content/drive/MyDrive/v11_chronos/abt_v10_cached.parquet'
OUT_DIR = '/content/drive/MyDrive/v11_chronos'
os.makedirs(OUT_DIR, exist_ok=True)

if not os.path.exists(ABT_PATH):
    raise FileNotFoundError(
        f'Cannot find {ABT_PATH}. Upload abt_v10_cached.parquet '
        f'to /content/drive/MyDrive/v11_chronos/ first.'
    )
abt = pd.read_parquet(ABT_PATH)
print('ABT shape:', abt.shape)
print('Columns sample:', list(abt.columns[:8]))
print(abt[['Период','Партнер','Артикул','target_qty_imputed']].head())

In [ ]:
abt['Период_p'] = pd.PeriodIndex(abt['Период'].astype(str), freq='M')
train_periods = pd.period_range('2020-01', '2024-06', freq='M')
val_periods   = pd.period_range('2024-07', '2025-06', freq='M')
test_periods  = pd.period_range('2025-07', '2026-01', freq='M')

active_pairs = (abt[abt['Период_p'].isin(val_periods) |
                    abt['Период_p'].isin(test_periods)]
                [['Партнер','Артикул']].drop_duplicates())
print('active pairs:', len(active_pairs))

wide = (abt.groupby(['Партнер','Артикул','Период_p'], observed=True)
             ['target_qty_imputed'].sum()
        .reset_index()
        .pivot_table(index=['Партнер','Артикул'], columns='Период_p',
                     values='target_qty_imputed', fill_value=0))
wide = wide.reindex(pd.MultiIndex.from_frame(active_pairs)).fillna(0).astype(np.float32)
print('wide shape:', wide.shape)

In [ ]:
from chronos import ChronosPipeline
device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype = torch.float16 if device == 'cuda' else torch.float32
print(f'Loading Chronos-T5-Small on {device} ({dtype}) ...')
pipe = ChronosPipeline.from_pretrained(
    'amazon/chronos-t5-small',
    device_map=device, torch_dtype=dtype,
)
print('OK')

In [ ]:
from tqdm.auto import tqdm

context_len = 36  # 3 years of monthly history
all_target_periods = list(val_periods) + list(test_periods)

predictions = {p: np.zeros(len(wide), dtype=np.float32)
               for p in all_target_periods}

rows = wide.values
col_to_ix = {c: i for i, c in enumerate(wide.columns)}
n_pairs = len(rows)

BATCH = 256

def context_for(pair_ix, target_period):
    end_ix = col_to_ix[target_period] - 1
    start_ix = max(0, end_ix - context_len + 1)
    return rows[pair_ix, start_ix:end_ix + 1]

t0 = time.time()
for tgt in tqdm(all_target_periods, desc='periods'):
    contexts = [context_for(i, tgt) for i in range(n_pairs)]
    for batch_start in range(0, n_pairs, BATCH):
        batch = [torch.tensor(c, dtype=torch.float32)
                 for c in contexts[batch_start:batch_start+BATCH]]
        with torch.no_grad():
            forecast = pipe.predict(context=batch, prediction_length=1, num_samples=20)
        median = np.median(forecast.cpu().numpy().astype(np.float32), axis=1).reshape(-1)
        median = np.clip(median, 0, None)
        predictions[tgt][batch_start:batch_start+len(batch)] = median
print(f'done forecasting in {(time.time()-t0)/60:.1f} min')

In [ ]:
out_rows = []
for tgt, preds in predictions.items():
    for i, (partner, sku) in enumerate(wide.index):
        out_rows.append({'Период': str(tgt), 'Партнер': partner, 'Артикул': sku,
                         'prediction': float(preds[i])})
df_out = pd.DataFrame(out_rows)

abt_lookup = abt[['Период','Партнер','Артикул','target_qty_imputed']].copy()
abt_lookup['Период'] = abt_lookup['Период'].astype(str)
df_out = df_out.merge(abt_lookup, on=['Период','Партнер','Артикул'], how='left')
df_out['target_qty'] = df_out['target_qty_imputed'].fillna(0)
df_out = df_out[['Период','Партнер','Артикул','target_qty','prediction']]

val_mask = df_out['Период'].isin([str(p) for p in val_periods])
test_mask = df_out['Период'].isin([str(p) for p in test_periods])

val_path = f'{OUT_DIR}/preds_v11_chronos_val.csv'
test_path = f'{OUT_DIR}/preds_v11_chronos_test.csv'
df_out[val_mask].to_csv(val_path, index=False)
df_out[test_mask].to_csv(test_path, index=False)
print(f'wrote {val_path} ({val_mask.sum()} rows)')
print(f'wrote {test_path} ({test_mask.sum()} rows)')

In [ ]:
def quick_metrics(df):
    a = df['target_qty'].to_numpy()
    p = df['prediction'].to_numpy()
    wape = float(np.abs(a - p).sum() / max(a.sum(), 1e-6))
    bias = float((p.sum() - a.sum()) / max(a.sum(), 1e-6) * 100)
    return {'rows': int(len(df)), 'WAPE': round(wape, 4),
            'bias_pct': round(bias, 2)}
print('VAL :', quick_metrics(df_out[val_mask]))
print('TEST:', quick_metrics(df_out[test_mask]))
print()
print('Chronos zero-shot done.  Download both CSVs from')
print(f'    {OUT_DIR}/preds_v11_chronos_val.csv')
print(f'    {OUT_DIR}/preds_v11_chronos_test.csv')
print('to your local repo at output/, then re-run scripts/v11_lad_stack.py')
print('Chronos will be picked up automatically by the LAD pool.')